In [7]:
import pandas as pd
from scipy.stats import pearsonr, spearmanr

In [12]:
import pandas as pd
from scipy.stats import pearsonr, spearmanr

def compute_agreement(filepath, likert=False):
    df = pd.read_csv(filepath)

    if likert:
        # rescale 1-7 -> 0-1 to match the other files' scale
        df["llama_typicality"] = (df["llama_typicality"] - 1) / 6

    sub = df.dropna(subset=["llama_typicality", "norm_rating_English"])

    pearson_r, pearson_p = pearsonr(sub["llama_typicality"], sub["norm_rating_English"])
    spearman_r, spearman_p = spearmanr(sub["llama_typicality"], sub["norm_rating_English"])

    return {
        "n": len(sub),
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_r": spearman_r,
        "spearman_p": spearman_p,
    }

files = [
    ("llama_en_logprob_prototypical.csv", False),
    ("llama_en_logprob.csv", False),
    ("llama_en_probability.csv", False),
    ("llama_en_likert.csv", True),   # needs 1-7 -> 0-1 normalization
]

for fname, is_likert in files:
    path = f"{fname}"
    result = compute_agreement(path, likert=is_likert)
    print(f"=== {fname} ===")
    print(f"n = {result['n']}")
    print(f"Pearson r = {result['pearson_r']:.4f} (p = {result['pearson_p']:.2e})")
    print(f"Spearman rho = {result['spearman_r']:.4f} (p = {result['spearman_p']:.2e})")
    print()

=== llama_en_logprob_prototypical.csv ===
n = 2355
Pearson r = 0.3817 (p = 1.46e-82)
Spearman rho = 0.5051 (p = 9.88e-153)

=== llama_en_logprob.csv ===
n = 2355
Pearson r = 0.3632 (p = 2.46e-74)
Spearman rho = 0.4528 (p = 2.04e-119)

=== llama_en_probability.csv ===
n = 2355
Pearson r = 0.3377 (p = 6.75e-64)
Spearman rho = 0.4388 (p = 2.09e-111)

=== llama_en_likert.csv ===
n = 2355
Pearson r = 0.3768 (p = 2.47e-80)
Spearman rho = 0.4328 (p = 3.92e-108)



In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv("llama_en_logprob_prototypical.csv")
sub = df.dropna(subset=["llama_typicality", "norm_rating_English"]).copy()

# 1. Confronto tra le medie/mediane generali
print("LLaMA  - mean:", sub["llama_typicality"].mean(), " median:", sub["llama_typicality"].median())
print("Human  - mean:", sub["norm_rating_English"].mean(), " median:", sub["norm_rating_English"].median())

# 2. Bias medio (differenza sistematica)
sub["diff"] = sub["llama_typicality"] - sub["norm_rating_English"]
print("\nMean diff (LLaMA - Human):", sub["diff"].mean())
# positivo => LLaMA tende a sovrastimare (troppo prototipico)
# negativo => LLaMA tende a sottostimare (troppo periferico)

# 3. Il bias dipende dal livello di tipicità umana?
# Divide gli item umani in bins (bassa/media/alta tipicità) e guarda il bias in ciascun bin
sub["human_bin"] = pd.qcut(sub["norm_rating_English"], 4, labels=["low", "mid-low", "mid-high", "high"])
print("\nBias medio per bin di tipicità umana:")
print(sub.groupby("human_bin")["diff"].mean())

# 4. Distribuzione dei valori LLaMA vicino agli estremi (0 e 1)
print("\n% valori LLaMA vicini a 1 (>0.9):", (sub["llama_typicality"] > 0.9).mean())
print("% valori LLaMA vicini a 0 (<0.1):", (sub["llama_typicality"] < 0.1).mean())
print("% valori umani vicini a 1 (>0.9):", (sub["norm_rating_English"] > 0.9).mean())
print("% valori umani vicini a 0 (<0.1):", (sub["norm_rating_English"] < 0.1).mean())

LLaMA  - mean: 0.7953649932738727  median: 0.9241418214862552
Human  - mean: 0.48469647635977875  median: 0.4745206714464973

Mean diff (LLaMA - Human): 0.310668516914094

Bias medio per bin di tipicità umana:
human_bin
low         0.434580
mid-low     0.387972
mid-high    0.284309
high        0.135767
Name: diff, dtype: float64

% valori LLaMA vicini a 1 (>0.9): 0.5825902335456475
% valori LLaMA vicini a 0 (<0.1): 0.05902335456475584
% valori umani vicini a 1 (>0.9): 0.02929936305732484
% valori umani vicini a 0 (<0.1): 0.021231422505307854


/tmp/ipykernel_5667/1481149418.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(sub.groupby("human_bin")["diff"].mean())
